In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/raw_cleaned")
OUTPUT_DIR = Path("output/reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILES = {
    "cuti": DATA_DIR / "cuti_standardized.xlsx",
    "tanggal": DATA_DIR / "raw_combined.xlsx",
}

KEY_COLUMNS = ["Employee ID", "Attendance Date"]


def profile_dataset(df, source_name):
    total_rows = len(df)
    records = []

    for column in df.columns:
        series = df[column]

        missing_count = series.isna().sum()
        unique_count = series.nunique(dropna=True)

        records.append({
            "source": source_name,
            "column_name": column,
            "data_type": str(series.dtype),
            "total_rows": total_rows,
            "missing_count": int(missing_count),
            "missing_percentage": round(
                missing_count / total_rows * 100, 2
            ) if total_rows else 0,
            "non_null_count": int(series.notna().sum()),
            "unique_count": int(unique_count),
            "sample_value": str(
                series.dropna().iloc[0]
            ) if series.notna().any() else None,
            "all_empty": bool(series.isna().all()),
        })

    return pd.DataFrame(records)


def summarize_dataset(df, source_name):
    valid_key = all(
        col in df.columns for col in KEY_COLUMNS
    )

    return {
        "source": source_name,
        "total_rows": len(df),
        "total_columns": len(df.columns),
        "unique_employees": (
            df["Employee ID"].nunique()
            if "Employee ID" in df.columns else None
        ),
        "duplicate_full_rows": int(df.duplicated().sum()),
        "duplicate_business_keys": (
            int(df.duplicated(KEY_COLUMNS).sum())
            if valid_key else None
        ),
        "all_empty_columns": int(df.isna().all().sum()),
    }


profiles = []
summaries = []
datasets = {}

for source_name, file_path in FILES.items():
    df = pd.read_excel(
        file_path,
        dtype={"Employee ID": "string"}
    )

    datasets[source_name] = df

    profiles.append(
        profile_dataset(df, source_name)
    )

    summaries.append(
        summarize_dataset(df, source_name)
    )

column_profile = pd.concat(
    profiles, ignore_index=True
)

dataset_summary = pd.DataFrame(summaries)

# Check overlapping business keys between files.
a = datasets["cuti"][KEY_COLUMNS].dropna().drop_duplicates()
b = datasets["tanggal"][KEY_COLUMNS].dropna().drop_duplicates()

overlap = a.merge(
    b,
    on=KEY_COLUMNS,
    how="inner"
)

relationship_summary = pd.DataFrame([{
    "shared_business_keys": len(overlap),
    "shared_column_names": len(
        set(datasets["cuti"].columns)
        & set(datasets["tanggal"].columns)
    ),
}])

output_file = OUTPUT_DIR / "data_assessment.xlsx"

with pd.ExcelWriter(output_file) as writer:
    column_profile.to_excel(
        writer, sheet_name="Column Profile", index=False
    )
    dataset_summary.to_excel(
        writer, sheet_name="Dataset Summary", index=False
    )
    relationship_summary.to_excel(
        writer, sheet_name="Relationship", index=False
    )

print(f"Profiling complete: {output_file}")


Profiling complete: output\reports\data_assessment.xlsx
